# Завдання
## Для даних, що оброблялись протягом лр1-4 реалізувати їх обробку з використанням алгоритмів ковзного вікна /апроксимації: Moving Average(MA), автокореляційні
алгоритми типу ARMA, ARIMA та інші:
## дослідити властивості даних;
## вибрати та реалізувати алгоритм обробки даних;
## реалізувати екстраполяцію даних на 0.5, 1.0, 1.5, 2 інтервалу спостереження;
## оцінити показники якості моделі та результатів прогнозування та екстраполяції;
## обґрунтувати вибір та ефективність етапів обробки;
## провести аналіз результатів розрахунків, довести їх придатність для використання.
# Група вимог_2:
## Провести комплекс R&D досліджень.
## Доповнити Групу вимог_1 рішеннями щодо оптимізації параметрів використаних
## алгоритмів ковзного вікна / апроксимації.

# Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

import warnings
warnings.filterwarnings("ignore")
import itertools

RANDOM_SEED = 42

# Metrics
def rmse(y, yhat): return np.sqrt(mean_squared_error(y, yhat))
def mae(y, yhat): return mean_absolute_error(y, yhat)
def mape(y, yhat): return np.mean(np.abs((y - yhat) / np.clip(y, 1e-8, None))) * 100


# Load data

In [ ]:
TARGET_COL = 'Estimated_Deliveries' # цільовий часовий ряд

df = pd.read_csv('data/tesla_deliveries_dataset_2015_2025.csv')
df['Date'] = pd.to_datetime(df[['Year','Month']].assign(Day=1))
df = df.sort_values(['Date']).reset_index(drop=True)

# Перевірка
print(df.info())
display(df.head())

# agg by date
series = df.groupby('Date')[TARGET_COL].sum().asfreq('MS')  # MS = month start
print("Series length:", len(series), "NaNs:", series.isna().sum())
series = series.interpolate()  # short_fill


# Decompose

In [ ]:
period = 12  # month slice
decomp = seasonal_decompose(series, model='additive', period=period, extrapolate_trend='freq')
fig = decomp.plot()
fig.set_size_inches(12,8)
plt.suptitle("Декомпозиція часової серії", fontsize=14)
plt.show()

# Короткі статистики
print("Series summary:")
display(series.describe())


# Season, autocorr

In [ ]:
def adf_test(x):
    res = adfuller(x.dropna())
    print(f"ADF statistic: {res[0]:.3f}, p-value: {res[1]:.4f}")

print("ADF test on original series:")
adf_test(series)

# ACF/PACF plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(series, ax=axes[0])
plot_pacf(series, ax=axes[1])


# Window optimization

In [ ]:
# MA functions + grid search
def forecast_ma(series_train, h, w):
    # last rolling mean value from train
    last_ma = series_train.rolling(window=w, min_periods=1).mean().iloc[-1]
    return np.repeat(last_ma, h)

def ts_cv_ma(series, window_grid=[3,6,12,24,36], n_splits=5, forecast_horizon=12):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    X = series.values
    best = None
    results = []
    for w in window_grid:
        rmses = []
        for train_idx, test_idx in tscv.split(X):
            # use last fold portion as forecast horizon
            train = pd.Series(X[train_idx])
            # forecast horizon = forecast_horizon or len(test_idx)
            h = min(forecast_horizon, len(test_idx))
            y_true = X[test_idx][:h]
            yhat = forecast_ma(train, h, w)
            rmses.append(rmse(y_true, yhat))
        results.append((w, np.mean(rmses)))
        if best is None or np.mean(rmses) < best[1]:
            best = (w, np.mean(rmses))
    return best, results

window_grid = [3,6,12,24,36]
best_w, res_grid = ts_cv_ma(series, window_grid=window_grid, n_splits=5, forecast_horizon=12)
print("MA grid results:", res_grid)
print("Best MA window:", best_w)

# ARIMA: підбір (p,d,q) та навчання — CV

In [ ]:
def evaluate_arima(order, train, test, h):
    # fit on train, forecast h steps
    try:
        model = ARIMA(train, order=order).fit()
        f = model.forecast(steps=h)
        return rmse(test[:h], f), f
    except Exception as e:
        return np.inf, None

def arima_grid_search(series, p_grid=[0,1,2], d_grid=[0,1], q_grid=[0,1,2], n_splits=3, h=12):
    orders = list(itertools.product(p_grid, d_grid, q_grid))
    tscv = TimeSeriesSplit(n_splits=n_splits)
    X = series.values
    best = (None, np.inf)
    order_scores = {}
    for order in orders:
        scores = []
        for train_idx, test_idx in tscv.split(X):
            train = X[train_idx]
            test = X[test_idx]
            score, _ = evaluate_arima(order, train, test, h=min(h,len(test)))
            scores.append(score)
            if np.mean(scores) > 1.5 * best[1]:  # early stop if already bad
                break
        mean_score = np.mean(scores) if scores else np.inf
        order_scores[order] = mean_score
        if mean_score < best[1]:
            best = (order, mean_score)
    return best, order_scores

p_grid=[0,1,2]; d_grid=[0,1]; q_grid=[0,1]
best_arima, arima_scores = arima_grid_search(series, p_grid, d_grid, q_grid, n_splits=3, h=12)
print("Best ARIMA order:", best_arima)


# Навчання фінальних моделей та екстраполяція на горизонти 0.5/1.0/1.5/2.0 *N

In [ ]:
# Final fit & multi-horizon forecasts
N = len(series)
horiz_factors = [0.5, 1.0, 1.5, 2.0]
horizons = [int(round(f * N)) for f in horiz_factors]
print("N:", N, "Horizons:", horizons)

# Fit final ARIMA with best_arima order
best_order = best_arima[0] if isinstance(best_arima, tuple) else best_arima
if best_order is None:
    best_order = (1,1,0)  # fallback
print("Using ARIMA order:", best_order)
arima_model = ARIMA(series, order=best_order).fit()

# Fit MA with best_w
w = best_w[0] if isinstance(best_w, tuple) else best_w
print("Using MA window:", w)

# produce forecasts
forecasts = {}
for h in horizons:
    # MA forecast
    ma_pred = forecast_ma(series, h, w)
    # ARIMA forecast
    arima_pred = arima_model.forecast(steps=h)
    forecasts[h] = {'MA': ma_pred, 'ARIMA': arima_pred}

# Save short sample prints
for h in horizons:
    print(f"\nHorizon {h} steps:")
    print("MA sample:", forecasts[h]['MA'][:5])
    print("ARIMA sample:", np.array(forecasts[h]['ARIMA'])[:5])


# Оцінка прогнозів

In [ ]:
# Evaluate short-term holdout
holdout = 12
train = series[:-holdout]
test = series[-holdout:]

# Fit ARIMA on train
model_short = ARIMA(train, order=best_order).fit()
arima_fore = model_short.forecast(steps=holdout)
ma_fore = forecast_ma(train, holdout, w)

print("Short-term holdout evaluation:")
print("ARIMA RMSE:", rmse(test.values, arima_fore))
print("ARIMA MAPE:", mape(test.values, arima_fore))
print("MA RMSE:", rmse(test.values, ma_fore))
print("MA MAPE:", mape(test.values, ma_fore))

# Plot
plt.figure(figsize=(12,5))
plt.plot(series.index, series.values, label='full series')
plt.plot(test.index, arima_fore, label='ARIMA forecast', marker='o')
plt.plot(test.index, ma_fore, label='MA forecast', marker='x')
plt.axvline(test.index[0], color='k', linestyle='--')
plt.legend(); plt.title('Holdout forecast comparison'); plt.show()


#  R&D: оптимізація параметрів ковзного вікна та ARIMA — порівняння KPI

In [ ]:
# R&D: fine grid search for MA window around best_w, and ARIMA (p up to 3)
ma_windows = list(range(max(2, w-5), w+6))
best_ma = (None, np.inf)
for ww in ma_windows:
    train = series[:-holdout]
    test = series[-holdout:]
    pred = forecast_ma(train, holdout, ww)
    s = rmse(test.values, pred)
    if s < best_ma[1]:
        best_ma = (ww, s)
print("R&D best MA window:", best_ma)

# ARIMA refinement limited
p_vals = [0,1,2,3]; d_vals = [0,1]; q_vals = [0,1,2]
best_ar = (None, np.inf)
for p in p_vals:
    for d in d_vals:
        for q in q_vals:
            try:
                model_t = ARIMA(train, order=(p,d,q)).fit()
                pred = model_t.forecast(steps=holdout)
                s = rmse(test.values, pred)
                if s < best_ar[1]:
                    best_ar = ((p,d,q), s)
            except Exception:
                continue
print("R&D best ARIMA:", best_ar)


#  Кластеризація часових рядів (мультифакторна): групуємо моделі/регіони за ознаками

In [ ]:
# Clustering by features per Model or Region
agg = df.groupby('Model').agg({
    'Avg_Price_USD':'mean',
    'Battery_Capacity_kWh':'mean',
    'Range_km':'mean',
    'Estimated_Deliveries':'sum'
}).dropna()

scaler = StandardScaler()
X = scaler.fit_transform(agg)
kmeans = KMeans(n_clusters=4, random_state=RANDOM_SEED).fit(X)
agg['Cluster'] = kmeans.labels_

# visualize
plt.figure(figsize=(8,6))
sns.scatterplot(data=agg, x='Range_km', y='Avg_Price_USD', hue='Cluster', palette='tab10')
plt.title("Кластерізація моделей EV")
plt.show()
display(agg.head())


# Кореляційний аналіз time series (по регіонах, моделях, лаги)

In [ ]:
# Correlation between series per Region
regional = df.groupby(['Region','Date'])[TARGET_COL].sum().unstack(level=0).fillna(0)
corr_regions = regional.corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr_regions, cmap='coolwarm', annot=False)
plt.title('Кореляція між регіонами')
plt.show()


#  Генерація синтетичних / модельних Time Series з виявленими властивостями

In [ ]:
# Synthesize series with similar properties
trend_comp = decomp.trend.fillna(method='bfill').fillna(method='ffill')
seasonal_comp = decomp.seasonal
resid_std = decomp.resid.std()

def synthesize_ts(trend, seasonal, resid_std, ar_phi=0.4, seed=RANDOM_SEED):
    np.random.seed(seed)
    N = len(trend)
    # AR(1) noise
    eps = np.zeros(N)
    for t in range(1,N):
        eps[t] = ar_phi * eps[t-1] + np.random.normal(scale=resid_std)
    synthetic = trend + seasonal + eps
    return synthetic

synthetic2 = synthesize_ts(trend_comp.values, seasonal_comp.values, resid_std, ar_phi=0.5)
plt.figure(figsize=(12,5))
plt.plot(series.index, series.values, label='original')
plt.plot(series.index, synthetic2, label='synthetic (AR noise)', alpha=0.8)
plt.legend(); plt.title('Original vs Synthetic (modelled)'); plt.show()

# Compare stats
print("Original mean/std:", series.mean(), series.std())
print("Synthetic mean/std:", synthetic2.mean(), synthetic2.std())
print("Corr:", np.corrcoef(series.values, synthetic2)[0,1])
